In [ ]:
import numpy as np

from scipy.integrate import solve_ivp
from scipy import sparse
from scipy.sparse.linalg import eigs


import plotly.graph_objects as go

# Get LORENT Stuff

In [2]:
sigma, rho, beta = 10, 28, 8.0/3.0

t_span = (0, 60)
t_steps = 6000
t = np.linspace(t_span[0], t_span[1], t_steps)

initial_state = [1.0, 1.0, 1.0]

In [3]:
def lorenz_system(t, state, sigma=sigma, rho=rho, beta=beta):
    x, y, z = state
    dx = sigma * (y - x)
    dy = x * (rho - z) - y
    dz = x * y - beta * z
    return [dx, dy, dz]

sol = solve_ivp(lorenz_system, t_span, initial_state, t_eval=t)

lorenz_dataset = sol.y.T

In [4]:
fig = go.Figure(
    data=go.Scatter3d(
        x=lorenz_dataset[:, 0],
        y=lorenz_dataset[:, 1],
        z=lorenz_dataset[:, 2],
        mode="lines",
        line=dict(color="blue", width=2),
    )
)

fig.show()

# Training

In [5]:
train_split = int(.8 * len(lorenz_dataset))

train_input = lorenz_dataset[0:train_split]
train_target = lorenz_dataset[1:train_split+1] # Basically just one over for future movement
test_input = lorenz_dataset[train_split:]

In [6]:
in_size = 3
out_size = 3 # In and out 3 since we have 3D data and next position output is also a 3D pos
res_size = 300
sparsity = .02
spec_rad = 1.2 # Need to test this out tbh
alpha = .3 # 70 percent previous and 30 percent current
ridge_weight = 1e-4 # Weight for ridge

In [ ]:
W_in = np.random.uniform(-0.1, 0.1, (res_size, in_size)) # Splits the 3 inputs to a 100 dims

W_sparse = sparse.random(
    res_size,
    res_size,
    density=sparsity,
    format="csr",
    data_rvs=lambda n: np.random.uniform(-1.0, 1.0, n),
)  # CSR is Compressed Sparse Row format also basically a Erdős–RényiErdős–Rényi graph

eigenvalues, _ = eigs(W_sparse, k=1, which="LM")
largest_eigenvalue = np.abs(eigenvalues[0])

W = W_sparse * (spec_rad / largest_eigenvalue)

In [ ]:
T_train = len(train_input)
X_states = np.zeros((res_size, T_train))
x = np.zeros(res_size)  # Initialize reservoir state vector to 0

for t in range(T_train):
    u_t = train_input[t]

    # Equation: x(t) = (1-alpha)*x(t-1) + alpha*tanh(W*x(t-1) + W_in*u(t))
    pre_activation = W.dot(x) + np.dot(W_in, u_t)
    x_target = np.tanh(pre_activation)
    x = (1 - alpha) * x + alpha * x_target

    X_states[:, t] = x

# Discard the initial transient states (Washout)
washout = 100
X_cleaned = X_states[:, washout:]
Y_target = train_target[washout:].T